# Novel proteoforms in PBMC and FTD/ALS (paper Supplementary Figures 7 and 8)

Origin: Houlin Yu's notebook (`MethodsDev/mdl_projects_2025`, `12272025_PBMC_Proteoform_Analysis/`, Jan 2026),
run then on the Feb-2026 ref-guided scg isoforms. This copy runs on the **LRAA v0.50.0** ref-guided,
sc cluster-guided isoforms of both eval trees, with the GENCODE v47 annotation those runs use.
ORFs: pyTD (PyTransDecoder) v0.4.0 via `__util/run_orfs.sh` (settings in `orf_settings.conf`): cDNA from gffread,
`-m 50`, one ORF per transcript, target-decoy q <= 0.05 for ORFs without homology support, homology retention
against `human.refdata-gex-GRCh38-GENCODE47.pep.fa` (diamond), start-codon refinement on; run 2026-11-04
(`ORFS_VERSION.txt` in each scg `data/` dir). Earlier versions of this analysis (LRAA v0.45.0, TransDecoder v6.0.0)
are parked in each `data/__ORFs_v0.45.0/`.

Run from this directory: `jupyter nbconvert --to notebook --execute --inplace <this notebook>`.

1. Cell 1: the analysis. For each dataset and minimum length (50, 100, 200 aa) it writes
   `<DS>_proteoform_analysis_results/novel_junction_results_protein_level_minAA<N>.tsv` and per-threshold plots.
2. Cell 2: the paper figures (`SuppFig_proteoforms_PBMC.pdf`, `SuppFig_proteoforms_FTD.pdf`; Supplementary Figures 7 and 8) and the counts
   behind them (`proteoform_counts.tsv`), from those tables.


In [ ]:
#!/usr/bin/env python3
"""
Novel Junction Analysis - Complete ORFs + Novel Junctions + Intron Retention
Gene-specific reference checking to avoid false positives from other genes.
"""

import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm import tqdm
import gzip
import matplotlib.pyplot as plt
import random
import os

# Dataset configurations: LRAA v0.51.0 ref-guided, sc cluster-guided isoforms (the eval trees' data/).
# PROTEIN_PEP / PROTEIN_GFF3: pyTD (PyTransDecoder) v0.4.0 on each GTF (__util/run_orfs.sh, settings in
# orf_settings.conf: -m 50, one ORF per transcript, decoy q <= 0.05, homology retention against the GENCODE47
# proteins, start refinement); complete ORFs are selected below.
REPO = os.path.abspath(os.path.join('..', '..'))  # the public repo root
PBMC_SCG = os.path.join(REPO, 'LRAA_paper-PBMCs_study/pbmcs_refGuided/pbmcs_refGuided_sc_cluster_guided')
ALS_SCG = os.path.join(REPO, 'LRAA_paper-FTD_study/als_refGuided/als_refGuided_sc_cluster_guided')

DATASETS = {
    'ALS': {
        'SQANTI_FILE': os.path.join(ALS_SCG, 'SQLIKE.LRAA.als.sc-cluster-guided.refGuided.iso_cats.tsv'),
        'PROTEIN_GFF3': os.path.join('orfs/FTD', 'LP00714.refGuided.scg.cdna.fasta.transdecoder.genome.gff3'),
        'PROTEIN_PEP': os.path.join('orfs/FTD', 'LP00714.refGuided.scg.cdna.fasta.transdecoder.pep'),
        'ISOFORM_GTF': os.path.join(ALS_SCG, 'data/LP00714.refguided.withGeneSymbols.gtf'),
        'OUTPUT_DIR': 'ALS_proteoform_analysis_results'
    },
    'PBMC': {
        'SQANTI_FILE': os.path.join(PBMC_SCG, 'SQLIKE.LRAA.pbmcs.sc-cluster-guided.refGuided.iso_cats.tsv'),
        'PROTEIN_GFF3': os.path.join('orfs/PBMC', 'PBMCs_pbio.LRAA.refGuided.scg.cdna.fasta.transdecoder.genome.gff3'),
        'PROTEIN_PEP': os.path.join('orfs/PBMC', 'PBMCs_pbio.LRAA.refGuided.scg.cdna.fasta.transdecoder.pep'),
        'ISOFORM_GTF': os.path.join(PBMC_SCG, 'data/PBMCs_pbio_CG_refguided.withGeneSymbols.gtf'),
        'OUTPUT_DIR': 'PBMC_proteoform_analysis_results'
    }
}

# Shared reference: the GENCODE v47 GTF the LRAA runs and their SQLIKE categories use. (The Jan-2026 run used the full
# gencode.v47.chr_patch_hapl_scaff GTF.)
REF_GTF = os.environ['REF_GTF']  # GENCODE v47 GTF (the reference the LRAA runs used)

for _ds in DATASETS.values():
    for _k in ('SQANTI_FILE', 'PROTEIN_GFF3', 'PROTEIN_PEP', 'ISOFORM_GTF'):
        assert os.path.exists(_ds[_k]), f"missing input {_k}: {_ds[_k]}"
assert os.path.exists(REF_GTF), REF_GTF

# Constants
AA_BUFFER = 0
NT_BUFFER = 0



def parse_gencode_gtf(gtf_file):
    """
    Parse GENCODE GTF to extract reference internal exons and splice sites PER GENE.
    Returns:
        ref_internal_exons: dict with key (gene_name, chrom, strand) -> set of (start, end) tuples
        ref_donors: dict with key (gene_name, chrom, strand) -> set of donor positions
        ref_acceptors: dict with key (gene_name, chrom, strand) -> set of acceptor positions
    """
    from collections import defaultdict
    
    ref_internal_exons = defaultdict(set)
    ref_donors = defaultdict(set)
    ref_acceptors = defaultdict(set)
    transcript_exons = defaultdict(list)
    transcript_info = {}  # Store gene_name, chrom, strand per transcript
    
    print("Parsing GENCODE GTF for reference exons and splice sites (gene-specific)...")
    
    open_func = gzip.open if gtf_file.endswith('.gz') else open
    try:
        with open_func(gtf_file, 'rt') as f:
            for line in tqdm(f, desc="Reading GENCODE GTF"):
                if line.startswith('#'):
                    continue
                
                fields = line.strip().split('\t')
                if len(fields) < 9:
                    continue
                
                feature = fields[2]
                if feature != 'exon':
                    continue
                
                chrom = fields[0]
                start = int(fields[3]) - 1  # Convert to 0-based
                end = int(fields[4])
                strand = fields[6]
                attributes = fields[8]
                
                # Extract gene_name and transcript_id
                gene_name = None
                transcript_id = None
                for attr in attributes.split(';'):
                    attr = attr.strip()
                    if attr.startswith('gene_name'):
                        gene_name = attr.split('"')[1]
                    elif attr.startswith('transcript_id'):
                        transcript_id = attr.split('"')[1]
                
                if not gene_name or not transcript_id:
                    continue
                
                # Store transcript info
                transcript_info[transcript_id] = (gene_name, chrom, strand)
                transcript_exons[transcript_id].append((start, end))
    except EOFError as e:
        print(f"\n⚠️  Warning: GTF file ended unexpectedly: {e}")
    
    # Process exons to identify internal exons and splice sites per gene
    print("Processing reference exons per gene...")
    for transcript_id, exons in tqdm(transcript_exons.items(), desc="Processing transcripts"):
        if transcript_id not in transcript_info:
            continue
        
        gene_name, chrom, strand = transcript_info[transcript_id]
        key = (gene_name, chrom, strand)
        
        # Sort exons
        exons_sorted = sorted(exons, key=lambda x: x[0])
        
        if len(exons_sorted) < 3:
            # Still collect splice sites even for transcripts with <3 exons
            for i, (exon_start, exon_end) in enumerate(exons_sorted):
                # Donors at exon ends (except last exon)
                if i < len(exons_sorted) - 1:
                    ref_donors[key].add(exon_end)
                # Acceptors at exon starts (except first exon)
                if i > 0:
                    ref_acceptors[key].add(exon_start)
            continue
        
        # Internal exons (exclude first and last)
        for exon in exons_sorted[1:-1]:
            ref_internal_exons[key].add(exon)
        
        # Collect all splice sites
        for i, (exon_start, exon_end) in enumerate(exons_sorted):
            # Donors at exon ends (except last exon)
            if i < len(exons_sorted) - 1:
                ref_donors[key].add(exon_end)
            # Acceptors at exon starts (except first exon)
            if i > 0:
                ref_acceptors[key].add(exon_start)
    
    print(f"Loaded reference data for {len(set(k[0] for k in ref_internal_exons.keys()))} genes")
    print(f"  - Internal exons: {sum(len(v) for v in ref_internal_exons.values()):,} total")
    print(f"  - Donor sites: {sum(len(v) for v in ref_donors.values()):,} total")
    print(f"  - Acceptor sites: {sum(len(v) for v in ref_acceptors.values()):,} total")
    
    return ref_internal_exons, ref_donors, ref_acceptors




def verify_coordinate_systems(dataset_config):
    """Sanity check: Verify all files use genomic coordinates (not transcript coordinates)."""
    print("\n" + "=" * 80)
    print("SANITY CHECK: Verifying Coordinate Systems")
    print("=" * 80)
    
    # Check Reference GTF
    print("\n1. Checking Reference GTF...")
    open_func = gzip.open if REF_GTF.endswith('.gz') else open
    with open_func(REF_GTF, 'rt') as f:
        for line in f:
            if not line.startswith('#') and line.strip():
                fields = line.strip().split('\t')
                if len(fields) >= 9 and fields[2] == 'exon':
                    print(f"   ✅ Reference GTF chromosome example: {fields[0]}")
                    print(f"   ✅ Genomic coordinates: {fields[3]}-{fields[4]}")
                    break
    
    # Check Isoform GTF
    print("\n2. Checking Isoform GTF...")
    ISOFORM_GTF = dataset_config['ISOFORM_GTF']
    open_func = gzip.open if ISOFORM_GTF.endswith('.gz') else open
    with open_func(ISOFORM_GTF, 'rt') as f:
        for line in f:
            if not line.startswith('#') and line.strip():
                fields = line.strip().split('\t')
                if len(fields) >= 9 and fields[2] == 'exon':
                    chrom = fields[0]
                    if chrom.startswith('chr') or chrom.isdigit():
                        print(f"   ✅ Isoform GTF chromosome example: {chrom}")
                        print(f"   ✅ Genomic coordinates: {fields[3]}-{fields[4]}")
                    else:
                        print(f"   ⚠️  WARNING: Isoform GTF may use transcript IDs: {chrom}")
                        print(f"   ⚠️  Expected chromosome names like 'chr1', got: {chrom}")
                    break
    
    # Check Protein GFF3
    print("\n3. Checking Protein GFF3...")
    PROTEIN_GFF3 = dataset_config['PROTEIN_GFF3']
    with open(PROTEIN_GFF3, 'r') as f:
        for line in f:
            if not line.startswith('#') and line.strip():
                fields = line.strip().split('\t')
                if len(fields) >= 9 and fields[2] == 'CDS':
                    chrom = fields[0]
                    if '.genome.gff3' in PROTEIN_GFF3:
                        print(f"   ✅ File name contains '.genome.gff3' (genomic coordinates)")
                    if chrom.startswith('chr') or chrom.isdigit():
                        print(f"   ✅ Protein GFF3 chromosome example: {chrom}")
                        print(f"   ✅ Genomic coordinates: {fields[3]}-{fields[4]}")
                    else:
                        print(f"   ❌ ERROR: Protein GFF3 uses transcript coordinates!")
                        print(f"   ❌ Found: {chrom} (should be chromosome like 'chr1')")
                        print(f"   ❌ Cannot proceed - coordinates won't match!")
                        return False
                    break
    
    print("\n" + "=" * 80)
    print("✅ Coordinate system verification complete - all files use genomic coordinates")
    print("=" * 80 + "\n")
    return True

def parse_protein_sequences(pep_file, min_aa_length=50):
    """
    Parse .pep file to get protein sequences and lengths for COMPLETE ORFs only.
    Apply minimum AA length filter.
    
    Args:
        pep_file: Path to .pep file
        min_aa_length: Minimum protein length in amino acids (default: 50)
    
    Returns: (complete_orf_proteins set, protein_lengths dict, protein_sequences dict)
    NOTE: Stop codons (*) are excluded from AA length count and sequences.
    """
    print(f"\nParsing protein sequences from .pep file: {pep_file}")
    print(f"Filtering to COMPLETE ORFs with ≥{min_aa_length} AA...")
    
    all_proteins = set()
    complete_orf_proteins = set()
    protein_lengths = {}  # protein_id -> AA length (excluding stop codon)
    protein_sequences = {}  # protein_id -> AA sequence (excluding stop codon)
    orf_type_counts = defaultdict(int)
    
    # Track filtering statistics
    complete_orf_count = 0
    length_filtered_count = 0
    
    current_protein_id = None
    current_sequence = []
    current_is_complete = False
    
    with open(pep_file, 'r') as f:
        for line in tqdm(f, desc="Reading .pep file"):
            line = line.strip()
            if line.startswith('>'):
                # Save previous protein if it was complete
                if current_protein_id and current_is_complete:
                    seq = ''.join(current_sequence)
                    # Remove stop codon (*) if present at the end
                    if seq.endswith('*'):
                        seq = seq[:-1]
                    aa_length = len(seq)
                    
                    # Apply length filter
                    if aa_length >= min_aa_length:
                        complete_orf_proteins.add(current_protein_id)
                        protein_lengths[current_protein_id] = aa_length
                        protein_sequences[current_protein_id] = seq
                    else:
                        length_filtered_count += 1
                
                # Reset for new protein
                current_sequence = []
                current_is_complete = False
                
                # Header line format: >PROTEIN_ID [rest of info]
                protein_id = line[1:].split()[0]  # Get first field after '>'
                current_protein_id = protein_id
                all_proteins.add(protein_id)
                
                # Extract ORF type from header
                if 'ORF type:' in line:
                    orf_part = line.split('ORF type:')[1]
                    orf_type = orf_part.split()[0].split(',')[0]
                    orf_type_counts[orf_type] += 1
                    
                    # Check if complete ORF
                    if orf_type == 'complete':
                        complete_orf_count += 1
                        current_is_complete = True
            else:
                # Sequence line
                if current_is_complete:
                    current_sequence.append(line)
        
        # Don't forget the last protein
        if current_protein_id and current_is_complete:
            seq = ''.join(current_sequence)
            # Remove stop codon (*) if present at the end
            if seq.endswith('*'):
                seq = seq[:-1]
            aa_length = len(seq)
            
            # Apply length filter
            if aa_length >= min_aa_length:
                complete_orf_proteins.add(current_protein_id)
                protein_lengths[current_protein_id] = aa_length
                protein_sequences[current_protein_id] = seq
            else:
                length_filtered_count += 1
    
    print(f"\n=== ORF Type Distribution ===")
    total = sum(orf_type_counts.values())
    for orf_type, count in sorted(orf_type_counts.items(), key=lambda x: -x[1]):
        print(f"  {orf_type:20s}: {count:6,} ({100*count/total:.1f}%)")
    
    print(f"\n✅ ORF Filtering Summary:")
    print(f"   Total proteins in .pep file:          {len(all_proteins):,}")
    print(f"   Complete ORF proteins:                {complete_orf_count:,}")
    print(f"   Complete ORFs with ≥{min_aa_length} AA:         {len(complete_orf_proteins):,}")
    print(f"   Filtered out (incomplete/partial):    {len(all_proteins) - complete_orf_count:,} ({100*(len(all_proteins)-complete_orf_count)/len(all_proteins):.1f}%)")
    print(f"   Filtered out (length <{min_aa_length} AA):       {length_filtered_count:,} ({100*length_filtered_count/complete_orf_count if complete_orf_count > 0 else 0:.1f}% of complete ORFs)")
    print(f"   Final protein count:                  {len(complete_orf_proteins):,}")
    print(f"   NOTE: Stop codons (*) excluded from AA length counts and sequences")
    
    # Show examples with lengths
    print(f"\n=== Complete ORF Protein Examples (≥{min_aa_length} AA) ===")
    for i, prot_id in enumerate(list(complete_orf_proteins)[:3]):
        aa_len = protein_lengths.get(prot_id, 'N/A')
        seq_preview = protein_sequences.get(prot_id, '')[:30] + '...' if len(protein_sequences.get(prot_id, '')) > 30 else protein_sequences.get(prot_id, '')
        print(f"  {i+1}. {prot_id} - {aa_len} AA")
        print(f"     Sequence: {seq_preview}")
    
    # Show length distribution
    if len(protein_lengths) > 0:
        lengths = list(protein_lengths.values())
        print(f"\n=== Protein Length Distribution (≥{min_aa_length} AA) ===")
        print(f"   Min:    {min(lengths)} AA")
        print(f"   Max:    {max(lengths)} AA")
        print(f"   Mean:   {np.mean(lengths):.1f} AA")
        print(f"   Median: {np.median(lengths):.1f} AA")
    
    return complete_orf_proteins, protein_lengths, protein_sequences

def build_reference_junction_set(gtf_file):
    """Build a set of all known junctions from reference GTF by reading exons directly."""
    print("\nBuilding reference junction set from GTF...")
    
    # Store exons grouped by transcript temporarily to extract junctions
    transcript_exons = defaultdict(list)
    
    open_func = gzip.open if gtf_file.endswith('.gz') else open
    try:
        with open_func(gtf_file, 'rt') as f:
            for line in tqdm(f, desc="Reading GTF exons"):
                if line.startswith('#') or line.strip() == '':
                    continue
                
                fields = line.strip().split('\t')
                if len(fields) < 9:
                    continue
                
                feature_type = fields[2]
                if feature_type != 'exon':
                    continue
                
                chrom = fields[0]
                start = int(fields[3]) - 1  # Convert to 0-based
                end = int(fields[4])  # GTF end is inclusive, becomes exclusive in 0-based
                strand = fields[6]
                attributes = fields[8]
                
                # Extract transcript_id
                transcript_id = None
                for attr in attributes.split(';'):
                    attr = attr.strip()
                    if attr.startswith('transcript_id'):
                        transcript_id = attr.split('"')[1]
                        break
                
                if transcript_id:
                    transcript_exons[transcript_id].append((chrom, strand, start, end))
    except EOFError as e:
        print(f"\n⚠️  Warning: GTF file ended unexpectedly (possibly truncated): {e}")
        print(f"Continuing with {len(transcript_exons)} transcripts read so far...")
    
    print(f"Found exons from {len(transcript_exons)} transcripts")
    
    # Extract all unique junctions
    ref_junctions = set()
    for transcript_id, exons in tqdm(transcript_exons.items(), desc="Extracting junctions"):
        # Sort exons by position
        exons.sort(key=lambda x: x[2])  # Sort by start position
        
        # Extract junctions between consecutive exons within the same transcript
        for i in range(len(exons) - 1):
            chrom1, strand1, start1, end1 = exons[i]
            chrom2, strand2, start2, end2 = exons[i + 1]
            
            # Ensure same chromosome and strand (same transcript/gene)
            if chrom1 == chrom2 and strand1 == strand2:
                donor = end1  # End of current exon
                acceptor = start2  # Start of next exon
                ref_junctions.add((chrom1, strand1, donor, acceptor))
    
    print(f"Built reference junction set with {len(ref_junctions)} unique junctions")
    
    # Debug: Show examples
    print("\n=== Reference Junction Examples ===")
    sample_junctions = list(ref_junctions)[:5]
    for junc in sample_junctions:
        print(f"  {junc}")
    print(f"\n=== Junction Data Structure ===")
    print(f"  Format: (chromosome, strand, donor_position, acceptor_position)")
    print(f"  Type: set of tuples for O(1) lookup")
    
    return ref_junctions

def parse_isoform_gtf(gtf_file):
    """Parse isoform GTF file and extract exon structures for each isoform."""
    print(f"\nParsing isoform GTF file: {gtf_file}")
    
    # Store exons grouped by transcript
    transcript_exons = defaultdict(list)
    
    open_func = gzip.open if gtf_file.endswith('.gz') else open
    with open_func(gtf_file, 'rt') as f:
        for line in tqdm(f, desc="Reading isoform GTF"):
            if line.startswith('#') or line.strip() == '':
                continue
            
            fields = line.strip().split('\t')
            if len(fields) < 9:
                continue
            
            feature_type = fields[2]
            if feature_type != 'exon':
                continue
            
            chrom = fields[0]
            start = int(fields[3]) - 1  # Convert to 0-based
            end = int(fields[4])  # GTF end is inclusive, becomes exclusive in 0-based
            strand = fields[6]
            attributes = fields[8]
            
            # Extract transcript_id
            transcript_id = None
            for attr in attributes.split(';'):
                attr = attr.strip()
                if attr.startswith('transcript_id'):
                    transcript_id = attr.split('"')[1]
                    break
            
            if transcript_id:
                transcript_exons[transcript_id].append((chrom, strand, start, end))
    
    # Sort exons for each transcript
    isoform_dict = {}
    for transcript_id, exons in transcript_exons.items():
        exons.sort(key=lambda x: x[2])  # Sort by start position
        
        # Get chromosome and strand from first exon
        chrom = exons[0][0]
        strand = exons[0][1]
        
        # Extract just the coordinates
        exon_coords = [(start, end) for _, _, start, end in exons]
        
        isoform_dict[transcript_id] = {
            'chrom': chrom,
            'strand': strand,
            'exons': exon_coords
        }
    
    print(f"Loaded {len(isoform_dict)} isoforms from GTF file")
    
    # Show example
    example_isoform = list(isoform_dict.keys())[0]
    print(f"\n=== Isoform Data Structure Example ===")
    print(f"Isoform ID: {example_isoform}")
    print(f"Data: {isoform_dict[example_isoform]}")
    
    return isoform_dict

def parse_gff3_protein_file(gff3_file, representative_proteins):
    """Parse GFF3 file from TransDecoder to extract protein CDS coordinates - ONLY complete ORF proteins."""
    print(f"\nParsing GFF3 file (filtering to complete ORF proteins only): {gff3_file}")
    protein_dict = defaultdict(lambda: {'chrom': None, 'strand': None, 'cds_regions': []})
    
    skipped_count = 0
    kept_count = 0
    
    with open(gff3_file, 'r') as f:
        for line in tqdm(f, desc="Reading GFF3"):
            if line.startswith('#') or line.strip() == '':
                continue
            
            fields = line.strip().split('\t')
            if len(fields) < 9:
                continue
            
            feature_type = fields[2]
            if feature_type != 'CDS':
                continue
            
            chrom = fields[0]
            start = int(fields[3]) - 1  # Convert to 0-based
            end = int(fields[4])  # GFF3 end is inclusive, becomes exclusive in 0-based
            strand = fields[6]
            attributes = fields[8]
            
            # Extract Parent transcript ID
            parent_id = None
            for attr in attributes.split(';'):
                if attr.startswith('Parent='):
                    parent_id = attr.split('=')[1]
                    break
            
            if parent_id:
                # FILTER: Only keep complete ORF proteins
                if parent_id not in representative_proteins:
                    skipped_count += 1
                    continue
                
                kept_count += 1
                protein_dict[parent_id]['chrom'] = chrom
                protein_dict[parent_id]['strand'] = strand
                protein_dict[parent_id]['cds_regions'].append((start, end))
    
    # Sort CDS regions for each protein
    for protein_id in protein_dict:
        protein_dict[protein_id]['cds_regions'].sort()
    
    print(f"Loaded {len(protein_dict)} complete ORF proteins from GFF3 file")
    print(f"  → Kept {kept_count:,} CDS regions from complete ORF proteins")
    print(f"  → Skipped {skipped_count:,} CDS regions from incomplete/partial ORFs")
    
    # Show example
    if len(protein_dict) > 0:
        example_protein = list(protein_dict.keys())[0]
        print(f"\n=== Protein Data Structure Example ===")
        print(f"Protein ID: {example_protein}")
        print(f"Data: {protein_dict[example_protein]}")
    
    return dict(protein_dict)

def extract_isoform_id_from_protein(protein_id):
    """Extract isoform ID from protein ID by removing .pX suffix."""
    if '.p' in protein_id:
        protein_id = protein_id.rsplit('.p', 1)[0]
    return protein_id

def extract_gene_name_from_isoform(isoform_id):
    """Extract gene name from isoform ID."""
    if '^' in isoform_id:
        return isoform_id.split('^')[0]
    return isoform_id

def build_protein_to_isoform_map(protein_data):
    """Pre-build a fast lookup from isoform_id to protein_id."""
    print("\nBuilding protein-to-isoform lookup map...")
    isoform_to_protein = {}
    
    for protein_id in tqdm(protein_data.keys(), desc="Mapping proteins"):
        isoform_id = extract_isoform_id_from_protein(protein_id)
        isoform_to_protein[isoform_id] = protein_id
    
    print(f"Mapped {len(isoform_to_protein)} isoforms to complete ORF proteins")
    
    # Show example
    if len(isoform_to_protein) > 0:
        example_isoform = list(isoform_to_protein.keys())[0]
        print(f"\n=== Isoform-to-Protein Mapping Example ===")
        print(f"Isoform ID: {example_isoform}")
        print(f"Protein ID: {isoform_to_protein[example_isoform]}")
    
    return isoform_to_protein

def detect_intron_retention(isoform_exons, ref_internal_exons, ref_donors, ref_acceptors, 
                           chrom, strand, gene_name, cds_regions, cds_start_overall, cds_end_overall):
    """
    Detect intron retention by finding internal exons NOT in gene-specific reference 
    but using known donor AND known acceptor sites FOR THAT GENE.
    
    Returns True if retained intron OVERLAPS with CDS buffer zone.
    
    HANDLES ALL CASES:
    1. Retained intron entirely within CDS buffer zone ✓
    2. Retained intron partially overlapping CDS buffer zone ✓
    3. Retained intron spanning entire CDS buffer zone ✓ (FIXED: now catches this!)
    4. Novel stop codons from boundary codons:
       - 2 bases from reference exon + 1 base from retained intron = stop codon ✓
    
    COORDINATE SYSTEM (0-based half-open):
    - exon_start: first base of exon (acceptor site position)
    - exon_end: first base after exon (donor site position)
    - Both define boundaries of retained intron
    
    BUFFER APPLICATION:
    - cds_end_overall INCLUDES stop codon (full CDS)
    - Buffer zone: [cds_start + NT_BUFFER, cds_end - NT_BUFFER]
    - Overlap check: exon_start < cds_buffer_end AND exon_end > cds_buffer_start
    
    NOTE: Stop codons can be formed at exon-intron boundaries:
    - Retained intron may introduce premature stop codons
    - These create novel proteoforms and are correctly detected
    """
    if len(isoform_exons) < 3:  # Need at least 3 exons to have internal exons
        return False
    
    # Get gene-specific reference key
    ref_key = (gene_name, chrom, strand)
    
    # Get reference sets for this gene (empty set if gene not in reference)
    gene_ref_internal_exons = ref_internal_exons.get(ref_key, set())
    gene_ref_donors = ref_donors.get(ref_key, set())
    gene_ref_acceptors = ref_acceptors.get(ref_key, set())
    
    # Calculate buffer zone boundaries (INCLUDES stop codon in cds_end_overall)
    cds_buffer_start = cds_start_overall + NT_BUFFER
    cds_buffer_end = cds_end_overall - NT_BUFFER
    
    # Check internal exons only (skip first and last)
    for i in range(1, len(isoform_exons) - 1):
        exon_start, exon_end = isoform_exons[i]
        
        # If this internal exon is NOT in gene-specific reference, check if it's intron retention
        if (exon_start, exon_end) not in gene_ref_internal_exons:
            # Check if BOTH endpoints use known splice sites FOR THIS GENE
            donor_known = exon_end in gene_ref_donors
            acceptor_known = exon_start in gene_ref_acceptors
            
            # Both must be known splice sites for intron retention
            if donor_known and acceptor_known:
                # This is intron retention - check OVERLAP with CDS buffer zone
                # 
                # OVERLAP LOGIC (catches ALL scenarios):
                # - If exon_start < cds_buffer_end: left boundary is before right edge of CDS
                # - If exon_end > cds_buffer_start: right boundary is after left edge of CDS
                # - Both conditions TRUE = some overlap exists
                #
                # This correctly handles:
                # ✓ Retention [150, 250] vs CDS buffer [100, 300]: 150 < 300 AND 250 > 100 = TRUE
                # ✓ Retention [50, 350] vs CDS buffer [100, 300]: 50 < 300 AND 350 > 100 = TRUE (spans entire CDS!)
                # ✓ Retention [250, 280] vs CDS buffer [100, 300]: 250 < 300 AND 280 > 100 = TRUE
                # ✗ Retention [50, 80] vs CDS buffer [100, 300]: 50 < 300 BUT 80 > 100 = FALSE (no overlap)
                # ✗ Retention [350, 400] vs CDS buffer [100, 300]: 350 < 300 = FALSE (no overlap)
                
                if exon_start < cds_buffer_end and exon_end > cds_buffer_start:
                    return True
    
    return False

def identify_novel_junctions(sqanti_file, protein_data, isoform_gtf_dict, isoform_to_protein, 
                            ref_junction_set, ref_internal_exons, ref_donors, ref_acceptors, 
                            protein_lengths, protein_sequences):
    """
    Main analysis function - Novel junctions AND intron retention detection with gene-specific reference checks.
    
    DEDUPLICATION STRATEGY:
    - Deduplicate by actual protein amino acid sequence (not CDS coordinates)
    - Proteins with identical sequences are considered the same, regardless of gene
    - Random selection based on proteoform ID (protein_id) for tie-breaking
    """
    print("\nLoading SQANTI classification file...")
    sqanti_df = pd.read_csv(sqanti_file, sep='\t')
    
    print(f"Total isoforms in SQANTI file: {len(sqanti_df)}")
    
    # Filter to only categories we care about
    all_categories = sqanti_df[sqanti_df['sqanti_cat'].isin(['FSM', 'ISM', 'NIC', 'NNIC'])].copy()
    print(f"Total FSM + ISM + NIC + NNIC isoforms: {len(all_categories)}")
    
    print(f"\n=== Novel Event Detection Criteria ===")
    print(f"Buffer requirement: ≥{AA_BUFFER} amino acids ({NT_BUFFER} nucleotides) from CDS boundaries")
    print(f"NOTE: CDS boundaries INCLUDE stop codon for junction/intron detection")
    print(f"\n1. Novel Junctions:")
    print(f"   - BOTH donor AND acceptor must be in CDS ± {NT_BUFFER}bp buffer")
    print(f"   - Strict inequalities (> and <) ensure positions INSIDE buffer zone")
    print(f"   - CDS boundaries: full CDS including stop codon")
    print(f"\n2. Intron Retention:")
    print(f"   - Internal exon NOT in reference (gene-specific)")
    print(f"   - BOTH endpoints use known splice sites (donor/acceptor) for same gene")
    print(f"   - Retained intron OVERLAPS with CDS ± {NT_BUFFER}bp buffer")
    print(f"   - Catches ALL cases including retention spanning entire CDS")
    print(f"   - CDS boundaries: full CDS including stop codon")
    
    # Track results
    all_results = []
    
    # Statistics tracking
    novel_junction_counts = []
    ref_junction_counts = []
    
    # Pre-extract the data we need from dataframe
    isoform_ids = all_categories['feature_name'].values
    categories = all_categories['sqanti_cat'].values
    
    print("\nAnalyzing isoforms...")
    for i in tqdm(range(len(all_categories)), desc="Processing isoforms"):
        isoform_id = isoform_ids[i]
        category = categories[i]
        
        # Get exon structure from GTF
        if isoform_id not in isoform_gtf_dict:
            continue
        
        gtf_data = isoform_gtf_dict[isoform_id]
        chrom = gtf_data['chrom']
        strand = gtf_data['strand']
        exon_list = gtf_data['exons']
        
        # Extract gene name from isoform ID
        gene_name = extract_gene_name_from_isoform(isoform_id)
        
        # Extract INTERNAL junctions only
        if len(exon_list) < 2:
            num_novel = 0
            num_ref_junctions = 0
            novel_junctions = []
        else:
            novel_junctions = []
            num_ref_junctions = 0
            
            for j in range(len(exon_list) - 1):
                donor = exon_list[j][1]
                acceptor = exon_list[j + 1][0]
                junction_key = (chrom, strand, donor, acceptor)
                
                if junction_key not in ref_junction_set:
                    novel_junctions.append((donor, acceptor))
                else:
                    num_ref_junctions += 1
            
            num_novel = len(novel_junctions)
        
        # Track statistics
        novel_junction_counts.append(num_novel)
        ref_junction_counts.append(num_ref_junctions)
        
        # Fast protein lookup (only complete ORF proteins)
        protein_id = isoform_to_protein.get(isoform_id)
        
        if protein_id is None:
            all_results.append({
                'isoform_id': isoform_id,
                'gene_name': gene_name,
                'protein_id': None,
                'protein_aa_length': None,
                'protein_sequence': None,
                'sqanti_cat': category,
                'chrom': chrom,
                'strand': strand,
                'num_exons': len(exon_list),
                'num_novel_junctions': num_novel,
                'num_ref_junctions': num_ref_junctions,
                'has_novel_junction_in_cds': False,
                'has_intron_retention_in_cds': False
            })
            continue
        
        # Get protein CDS regions
        prot_info = protein_data[protein_id]
        cds_regions = prot_info['cds_regions']
        
        # Get protein sequence for deduplication
        protein_seq = protein_sequences.get(protein_id, '')
        
        # Get total CDS boundaries - INCLUDE stop codon (full CDS)
        cds_start_overall = cds_regions[0][0]
        cds_end_overall = cds_regions[-1][1]
        
        # Calculate buffer zone boundaries ONCE
        cds_buffer_start = cds_start_overall + NT_BUFFER
        cds_buffer_end = cds_end_overall - NT_BUFFER
        
        # NOVEL JUNCTION DETECTION
        has_novel_junction_in_cds = False
        if num_novel > 0:
            for donor, acceptor in novel_junctions:
                donor_in_cds_valid = False
                acceptor_in_cds_valid = False
                
                # Check if donor is within any CDS region
                for cds_start, cds_end in cds_regions:
                    if cds_start <= donor <= cds_end:
                        if donor > cds_buffer_start and donor < cds_buffer_end:
                            donor_in_cds_valid = True
                            break
                
                # Check if acceptor is within any CDS region
                for cds_start, cds_end in cds_regions:
                    if cds_start <= acceptor <= cds_end:
                        if acceptor > cds_buffer_start and acceptor < cds_buffer_end:
                            acceptor_in_cds_valid = True
                            break
                
                if donor_in_cds_valid and acceptor_in_cds_valid:
                    has_novel_junction_in_cds = True
                    break
        
        # INTRON RETENTION DETECTION
        has_intron_retention = detect_intron_retention(
            exon_list, ref_internal_exons, ref_donors, ref_acceptors, 
            chrom, strand, gene_name, cds_regions, cds_start_overall, cds_end_overall
        )
        
        all_results.append({
            'isoform_id': isoform_id,
            'gene_name': gene_name,
            'protein_id': protein_id,
            'protein_aa_length': protein_lengths.get(protein_id),
            'protein_sequence': protein_seq,
            'sqanti_cat': category,
            'chrom': chrom,
            'strand': strand,
            'num_exons': len(exon_list),
            'num_novel_junctions': num_novel,
            'num_ref_junctions': num_ref_junctions,
            'has_novel_junction_in_cds': has_novel_junction_in_cds,
            'has_intron_retention_in_cds': has_intron_retention
        })
    
    results_df = pd.DataFrame(all_results)
    
    print(f"\n=== Results DataFrame Info ===")
    print(f"Shape: {results_df.shape}")
    print(f"Isoforms with complete ORF proteins: {results_df['protein_id'].notna().sum():,}")
    print(f"Isoforms without complete ORF proteins: {results_df['protein_id'].isna().sum():,}")
    
    # DEDUPLICATION BY PROTEIN SEQUENCE
    print("\n" + "=" * 80)
    print("DEDUPLICATION BY PROTEIN AMINO ACID SEQUENCE")
    print("=" * 80)
    
    # Group ALL proteins by sequence
    proteins_with_data = results_df[results_df['protein_id'].notna()].copy()
    
    print(f"\nTotal proteins before deduplication: {len(proteins_with_data):,}")
    
    sequence_groups = proteins_with_data.groupby('protein_sequence')
    
    print(f"Total unique protein sequences: {len(sequence_groups)}")
    
    deduplicated_results = []
    dedup_stats = {
        'total_unique_sequences': 0,
        'matched': 0,
        'novel_junction_only': 0,
        'intron_retention_only': 0,
        'both_events': 0,
        'reclassified_to_matched': 0,
        'multi_gene_proteins': 0  # Track proteins matching multiple genes
    }
    
    for protein_seq, group in tqdm(sequence_groups, desc="Deduplicating proteins"):
        dedup_stats['total_unique_sequences'] += 1
        
        # Check if this sequence maps to multiple genes
        unique_genes = group['gene_name'].nunique()
        if unique_genes > 1:
            dedup_stats['multi_gene_proteins'] += 1

        # Check if ANY isoform in this group is "Matched" (no novel events)
        has_matched_isoform = ((~group['has_novel_junction_in_cds']) & 
                               (~group['has_intron_retention_in_cds'])).any()
        
        if has_matched_isoform:
            # At least one isoform produces this protein WITHOUT novel events
            # So novel events in other isoforms don't create novel proteoforms
            # Pick one random isoform (by protein_id for reproducibility) but force classification as "Matched"
            selected = group.sort_values('protein_id').sample(n=1, random_state=42).iloc[0].to_dict()
            
            # Check if this selected one had novel events (for tracking)
            had_novel_events = (selected['has_novel_junction_in_cds'] or 
                               selected['has_intron_retention_in_cds'])
            
            # Force reclassification to Matched
            selected['has_novel_junction_in_cds'] = False
            selected['has_intron_retention_in_cds'] = False
            
            dedup_stats['matched'] += 1
            if had_novel_events:
                dedup_stats['reclassified_to_matched'] += 1
        else:
            # No matched isoform exists - novel events are real
            # Pick ONE random isoform (by protein_id for reproducibility) and keep its classification
            selected = group.sort_values('protein_id').sample(n=1, random_state=42).iloc[0].to_dict()
            
            has_nj = selected['has_novel_junction_in_cds']
            has_ir = selected['has_intron_retention_in_cds']
            
            if has_nj and not has_ir:
                dedup_stats['novel_junction_only'] += 1
            elif not has_nj and has_ir:
                dedup_stats['intron_retention_only'] += 1
            else:  # both
                dedup_stats['both_events'] += 1

        deduplicated_results.append(selected)
        
    # Add back proteins without protein_id
    no_protein = results_df[results_df['protein_id'].isna()]
    for _, row in no_protein.iterrows():
        deduplicated_results.append(row.to_dict())
    
    # Create deduplicated dataframe
    results_dedup_df = pd.DataFrame(deduplicated_results)
    
    # Add is_representative flag
    results_dedup_df['is_representative'] = results_dedup_df['protein_id'].notna()
    
    print(f"\n=== Deduplication Summary ===")
    print(f"Total proteins before deduplication: {len(proteins_with_data):,}")
    print(f"Unique protein sequences: {dedup_stats['total_unique_sequences']:,}")
    print(f"Proteins matching multiple genes: {dedup_stats['multi_gene_proteins']:,}")
    print(f"\nNovel event classification:")
    print(f"  - Matched (no novel events):      {dedup_stats['matched']:,}")
    print(f"    (including {dedup_stats['reclassified_to_matched']:,} reclassified)")
    print(f"  - Novel junction only:            {dedup_stats['novel_junction_only']:,}")
    print(f"  - Intron retention only:          {dedup_stats['intron_retention_only']:,}")
    print(f"  - Both events:                    {dedup_stats['both_events']:,}")
    
    # Print statistics
    print("\n=== Analysis Summary ===")
    print(f"Total isoforms analyzed: {len(results_dedup_df)}")
    print(f"Isoforms with complete ORF proteins: {results_dedup_df['protein_id'].notna().sum()}")
    print(f"Unique proteins (deduplicated by sequence): {results_dedup_df[results_dedup_df['is_representative']].shape[0]}")
    
    # Show protein AA length statistics
    proteins_with_length = results_dedup_df[results_dedup_df['protein_aa_length'].notna()]
    if len(proteins_with_length) > 0:
        print(f"\n=== Protein AA Length Statistics ===")
        print(f"Proteins with AA length: {len(proteins_with_length):,}")
        print(f"Mean AA length: {proteins_with_length['protein_aa_length'].mean():.1f}")
        print(f"Median AA length: {proteins_with_length['protein_aa_length'].median():.1f}")
        print(f"Min AA length: {int(proteins_with_length['protein_aa_length'].min())}")
        print(f"Max AA length: {int(proteins_with_length['protein_aa_length'].max())}")
    
    # Protein-level statistics by category
    protein_results = results_dedup_df[results_dedup_df['is_representative']].copy()
    print(f"\n=== Protein-Level Results by SQANTI Category ===")
    for cat in ['FSM', 'ISM', 'NIC', 'NNIC']:
        cat_prots = protein_results[protein_results['sqanti_cat'] == cat]
        total = len(cat_prots)
        if total > 0:
            matched = len(cat_prots[(~cat_prots['has_novel_junction_in_cds']) & 
                                   (~cat_prots['has_intron_retention_in_cds'])])
            novel_junc = len(cat_prots[(cat_prots['has_novel_junction_in_cds']) & 
                                      (~cat_prots['has_intron_retention_in_cds'])])
            intron_ret = len(cat_prots[(~cat_prots['has_novel_junction_in_cds']) & 
                                      (cat_prots['has_intron_retention_in_cds'])])
            both = len(cat_prots[(cat_prots['has_novel_junction_in_cds']) & 
                                (cat_prots['has_intron_retention_in_cds'])])
            
            print(f"{cat:4s}: {total:6,} proteins | Matched: {matched:5,} | "
                  f"NJ: {novel_junc:5,} | IR: {intron_ret:5,} | Both: {both:5,}")
    
    return results_dedup_df

def subsample_proteins_by_gene(results_df, n_per_group=5, min_isoforms=2, max_isoforms=4, output_dir='.', suffix=''):
    """
    Subsample proteins from NIC and NNIC categories - updated for 4 categories.
    
    Args:
        results_df: Results dataframe
        n_per_group: Number of proteins per group
        min_isoforms: Minimum number of isoforms per gene
        max_isoforms: Maximum number of isoforms per gene
        output_dir: Output directory
        suffix: Suffix for output filename (e.g., '_minAA200')
    """
    print("\n" + "=" * 80)
    print("SUBSAMPLING PROTEINS FROM GENES WITH 2-4 ISOFORMS")
    print(f"Target: {n_per_group} per group × 4 categories = {n_per_group * 4} total proteins")
    print("=" * 80)
    
    # Filter to representative proteins only
    rep_proteins = results_df[results_df['is_representative'] == True].copy()
    
    # Count isoforms per gene
    gene_isoform_counts = results_df.groupby('gene_name')['isoform_id'].nunique().reset_index()
    gene_isoform_counts.columns = ['gene_name', 'total_isoforms']
    
    # Filter genes with 2-4 isoforms
    valid_genes = gene_isoform_counts[
        (gene_isoform_counts['total_isoforms'] >= min_isoforms) & 
        (gene_isoform_counts['total_isoforms'] <= max_isoforms)
    ]['gene_name'].tolist()
    
    print(f"\nFound {len(valid_genes)} genes with {min_isoforms}-{max_isoforms} isoforms")
    
    # Filter representative proteins from valid genes
    rep_proteins_valid_genes = rep_proteins[rep_proteins['gene_name'].isin(valid_genes)].copy()
    
    # Add isoform count to the dataframe
    rep_proteins_valid_genes = rep_proteins_valid_genes.merge(
        gene_isoform_counts, on='gene_name', how='left'
    )
    
    # Create novel event category column
    def categorize_novel_event(row):
        has_nj = row['has_novel_junction_in_cds']
        has_ir = row['has_intron_retention_in_cds']
        if not has_nj and not has_ir:
            return 'Matched'
        elif has_nj and not has_ir:
            return 'NJ_only'
        elif not has_nj and has_ir:
            return 'IR_only'
        else:
            return 'Both'
    
    rep_proteins_valid_genes['novel_event_category'] = rep_proteins_valid_genes.apply(categorize_novel_event, axis=1)
    
    # Subsample from NIC/NNIC combined, grouped by novel event category
    all_samples = []
    nic_nnic = rep_proteins_valid_genes[rep_proteins_valid_genes['sqanti_cat'].isin(['NIC', 'NNIC'])]
    
    for event_cat in ['Matched', 'NJ_only', 'IR_only', 'Both']:
        cat_proteins = nic_nnic[nic_nnic['novel_event_category'] == event_cat]
        print(f"\n{event_cat}: {len(cat_proteins)} available")
        if len(cat_proteins) >= n_per_group:
            sample = cat_proteins.sample(n=n_per_group, random_state=42)
            print(f"   → Sampled {n_per_group} proteins")
        else:
            sample = cat_proteins
            print(f"   ⚠️  Only {len(cat_proteins)} proteins available (requested {n_per_group})")
        all_samples.append(sample)
    
    # Combine all samples
    subsample_df = pd.concat(all_samples, ignore_index=True)
    
    # Select relevant columns for output
    output_columns = [
        'gene_name',
        'total_isoforms',
        'isoform_id',
        'protein_id',
        'protein_aa_length',
        'sqanti_cat',
        'novel_event_category',
        'has_novel_junction_in_cds',
        'has_intron_retention_in_cds',
        'chrom',
        'strand',
        'num_exons',
        'num_novel_junctions',
        'num_ref_junctions'
    ]
    
    subsample_output = subsample_df[output_columns].copy()
    
    # Sort by novel event category and gene name
    subsample_output = subsample_output.sort_values(['novel_event_category', 'gene_name'], 
                                                     ascending=[True, True])
    
    # Save to file in output directory with suffix
    output_file = os.path.join(output_dir, f'subsampled_proteins_for_review{suffix}.tsv')
    subsample_output.to_csv(output_file, sep='\t', index=False)
    
    print(f"\n=== Subsampled Proteins Summary ===")
    print(f"Total proteins sampled: {len(subsample_output)}")
    for event_cat in ['Matched', 'NJ_only', 'IR_only', 'Both']:
        count = len(subsample_output[subsample_output['novel_event_category'] == event_cat])
        print(f"  - {event_cat:15s}: {count}")
    
    # Print AA length statistics
    print(f"\n=== Protein AA Length Statistics ===")
    if subsample_output['protein_aa_length'].notna().sum() > 0:
        print(f"  Min: {subsample_output['protein_aa_length'].min():.0f} AA")
        print(f"  Max: {subsample_output['protein_aa_length'].max():.0f} AA")
        print(f"  Mean: {subsample_output['protein_aa_length'].mean():.1f} AA")
        print(f"  Median: {subsample_output['protein_aa_length'].median():.1f} AA")
    
    print(f"\nSaved to: {output_file}")
    
    # Print the subsampled proteins in a nice table
    print("\n" + "=" * 80)
    print("SUBSAMPLED PROTEINS FOR REVIEW")
    print("=" * 80)
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)
    print(subsample_output.to_string(index=False))
    print("=" * 80)
    
    return subsample_output

def plot_transcript_level_counts(sqanti_file, output_prefix='transcript_level_counts'):
    """Create simple bar plot of transcript counts by SQANTI category."""
    print("\n=== Creating Transcript-Level Plot (No Deduplication) ===")
    
    sqanti_df = pd.read_csv(sqanti_file, sep='\t')
    category_counts = sqanti_df['sqanti_cat'].value_counts()
    
    categories = ['FSM', 'ISM', 'NIC', 'NNIC']
    counts = [category_counts.get(cat, 0) for cat in categories]
    
    print(f"\n=== Transcript-Level Counts (Raw SQANTI) ===")
    for cat, count in zip(categories, counts):
        print(f"{cat:4s}: {count:6,} transcripts")
    print(f"{'TOTAL':4s}: {sum(counts):6,} transcripts")
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    x_pos = np.arange(len(categories))
    colors = ['#3498db', '#9b59b6', '#e67e22', '#e74c3c']
    
    bars = ax.bar(x_pos, counts, width=0.6, color=colors, edgecolor='black', linewidth=1.5)
    
    for i, (bar, count) in enumerate(zip(bars, counts)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{count:,}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    ax.set_xlabel('SQANTI Category', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Transcripts', fontsize=14, fontweight='bold')
    ax.set_title('Transcript Counts by SQANTI Category\n(No Deduplication, No Filtering)', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(categories, fontsize=12)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    legend_text = (
        'FSM: Full Splice Match\n'
        'ISM: Incomplete Splice Match\n'
        'NIC: Novel In Catalog\n'
        'NNIC: Novel Not In Catalog'
    )
    ax.text(0.98, 0.97, legend_text, transform=ax.transAxes,
            fontsize=10, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    
    # Save in three formats: PNG, SVG, and PDF
    png_file = f'{output_prefix}.png'
    svg_file = f'{output_prefix}.svg'
    pdf_file = f'{output_prefix}.pdf'
    plt.savefig(png_file, dpi=300, bbox_inches='tight')
    plt.savefig(svg_file, format='svg', bbox_inches='tight')
    plt.savefig(pdf_file, format='pdf', bbox_inches='tight')
    print(f"\nVisualization saved to:")
    print(f"  - {png_file}")
    print(f"  - {svg_file}")
    print(f"  - {pdf_file}")
    plt.close()
    
def plot_results(results_df, output_prefix='novel_junction_analysis', min_aa_length=None):
    """
    Create visualization of results - 7 plots: 1 bar + 6 pie charts (3 for all, 3 for novel proteoforms only).
    
    Args:
        results_df: Results dataframe
        output_prefix: Output file prefix
        min_aa_length: Minimum AA length threshold (for plot title)
    """
    
    rep_proteins = results_df[results_df['is_representative'] == True].copy()
    
    print(f"\n=== Plotting Statistics ===")
    print(f"Total rows in results: {len(results_df)}")
    print(f"Representative proteins only: {len(rep_proteins)}")
    
    # Classify proteins into 4 categories
    category_counts = {}
    for cat in ['FSM', 'ISM', 'NIC', 'NNIC']:
        cat_data = rep_proteins[rep_proteins['sqanti_cat'] == cat]
        
        matched = len(cat_data[(~cat_data['has_novel_junction_in_cds']) & 
                              (~cat_data['has_intron_retention_in_cds'])])
        novel_junc_only = len(cat_data[(cat_data['has_novel_junction_in_cds']) & 
                                      (~cat_data['has_intron_retention_in_cds'])])
        intron_ret_only = len(cat_data[(~cat_data['has_novel_junction_in_cds']) & 
                                      (cat_data['has_intron_retention_in_cds'])])
        both_events = len(cat_data[(cat_data['has_novel_junction_in_cds']) & 
                                  (cat_data['has_intron_retention_in_cds'])])
        
        category_counts[cat] = {
            'matched': matched,
            'novel_junction': novel_junc_only,
            'intron_retention': intron_ret_only,
            'both': both_events,
            'total': len(cat_data)
        }
    
    # Create figure with 7 subplots (2 rows, 4 columns)
    fig = plt.figure(figsize=(20, 10))
    gs = fig.add_gridspec(2, 4, hspace=0.3, wspace=0.3)
    
    ax1 = fig.add_subplot(gs[0, 0])  # Bar chart
    ax2 = fig.add_subplot(gs[0, 1])  # NIC all events
    ax3 = fig.add_subplot(gs[0, 2])  # NNIC all events
    ax4 = fig.add_subplot(gs[0, 3])  # Combined all events
    ax5 = fig.add_subplot(gs[1, 1])  # NIC novel proteoforms only
    ax6 = fig.add_subplot(gs[1, 2])  # NNIC novel proteoforms only
    ax7 = fig.add_subplot(gs[1, 3])  # Combined novel proteoforms only
    
    # Panel 1: Stacked bar - Novel Proteoforms (combined) vs Others (Matched)
    categories = ['FSM', 'ISM', 'NIC', 'NNIC']
    x_pos = np.arange(len(categories))
    width = 0.6
    
    matched_counts = [category_counts[cat]['matched'] for cat in categories]
    # Combine all three novel event types into "Novel Proteoforms"
    novel_proteoforms_counts = [
        category_counts[cat]['novel_junction'] + 
        category_counts[cat]['intron_retention'] + 
        category_counts[cat]['both'] 
        for cat in categories
    ]
    
    # Color scheme - simplified to 2 colors
    color_matched = '#95a5a6'  # Gray - Others
    color_novel = '#e74c3c'  # Red - Novel Proteoforms
    
    p1 = ax1.bar(x_pos, matched_counts, width, label='Others', color=color_matched)
    p2 = ax1.bar(x_pos, novel_proteoforms_counts, width, bottom=matched_counts, 
                 label='Novel Proteoforms', color=color_novel)
    
    ax1.set_xlabel('SQANTI Category', fontsize=12)
    ax1.set_ylabel('Number of Unique Proteins', fontsize=12)
    
    # Add min_aa_length to title if provided
    title = 'Protein Counts by Category\n(Deduplicated, Complete ORFs'
    if min_aa_length is not None:
        title += f', ≥{min_aa_length} AA'
    title += ')'
    ax1.set_title(title, fontsize=14, fontweight='bold')
    
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(categories)
    ax1.legend(fontsize=10, loc='upper right')
    ax1.grid(axis='y', alpha=0.3)
    
    # Add numbers inside bars
    min_height_for_text = ax1.get_ylim()[1] * 0.03
    for i, cat in enumerate(categories):
        if matched_counts[i] > min_height_for_text:
            ax1.text(i, matched_counts[i]/2, f'{matched_counts[i]:,}', 
                    ha='center', va='center', fontsize=10, fontweight='bold', color='white')
        if novel_proteoforms_counts[i] > min_height_for_text:
            ax1.text(i, matched_counts[i] + novel_proteoforms_counts[i]/2, 
                    f'{novel_proteoforms_counts[i]:,}', 
                    ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    
    # Color scheme for pie charts (4 categories for all events)
    color_matched_pie = '#95a5a6'  # Gray
    color_nj = '#3498db'  # Blue
    color_ir = '#f39c12'  # Orange
    color_both = '#e74c3c'  # Red
    
    # Color scheme for novel proteoforms only (3 categories)
    color_nj_novel = '#3498db'  # Blue
    color_ir_novel = '#f39c12'  # Orange
    color_both_novel = '#e74c3c'  # Red
    
    # Panel 2: Pie chart for NIC (all events)
    nic_counts = category_counts['NIC']
    nic_matched = nic_counts['matched']
    nic_nj = nic_counts['novel_junction']
    nic_ir = nic_counts['intron_retention']
    nic_both = nic_counts['both']
    nic_total = nic_counts['total']
    
    nic_sizes = [nic_matched, nic_nj, nic_ir, nic_both]
    nic_labels = [
        f'Others\n({nic_matched:,})',
        f'Novel Junction\n({nic_nj:,})',
        f'Intron Retention\n({nic_ir:,})',
        f'NJ + IR\n({nic_both:,})'
    ]
    nic_colors = [color_matched_pie, color_nj, color_ir, color_both]
    
    # Filter out zero values
    nic_sizes_filtered = [s for s in nic_sizes if s > 0]
    nic_labels_filtered = [l for l, s in zip(nic_labels, nic_sizes) if s > 0]
    nic_colors_filtered = [c for c, s in zip(nic_colors, nic_sizes) if s > 0]
    
    wedges2, texts2, autotexts2 = ax2.pie(nic_sizes_filtered, labels=nic_labels_filtered, 
                                           colors=nic_colors_filtered, 
                                           autopct='%1.1f%%', startangle=90,
                                           textprops={'fontsize': 14},
                                           pctdistance=0.85)
    
    for autotext in autotexts2:
        autotext.set_fontsize(12)
        autotext.set_fontweight('bold')
        autotext.set_color('white')
    
    for text in texts2:
        text.set_fontsize(14)
        text.set_fontweight('bold')
    
    ax2.set_title(f'NIC - All Events\n(Total: {nic_total:,} proteins)', 
                  fontsize=14, fontweight='bold')
    
    # Panel 3: Pie chart for NNIC (all events)
    nnic_counts = category_counts['NNIC']
    nnic_matched = nnic_counts['matched']
    nnic_nj = nnic_counts['novel_junction']
    nnic_ir = nnic_counts['intron_retention']
    nnic_both = nnic_counts['both']
    nnic_total = nnic_counts['total']
    
    nnic_sizes = [nnic_matched, nnic_nj, nnic_ir, nnic_both]
    nnic_labels = [
        f'Others\n({nnic_matched:,})',
        f'Novel Junction\n({nnic_nj:,})',
        f'Intron Retention\n({nnic_ir:,})',
        f'NJ + IR\n({nnic_both:,})'
    ]
    nnic_colors = [color_matched_pie, color_nj, color_ir, color_both]
    
    # Filter out zero values
    nnic_sizes_filtered = [s for s in nnic_sizes if s > 0]
    nnic_labels_filtered = [l for l, s in zip(nnic_labels, nnic_sizes) if s > 0]
    nnic_colors_filtered = [c for c, s in zip(nnic_colors, nnic_sizes) if s > 0]
    
    wedges3, texts3, autotexts3 = ax3.pie(nnic_sizes_filtered, labels=nnic_labels_filtered, 
                                           colors=nnic_colors_filtered, 
                                           autopct='%1.1f%%', startangle=90,
                                           textprops={'fontsize': 14},
                                           pctdistance=0.85)
    
    for autotext in autotexts3:
        autotext.set_fontsize(12)
        autotext.set_fontweight('bold')
        autotext.set_color('white')
    
    for text in texts3:
        text.set_fontsize(14)
        text.set_fontweight('bold')
    
    ax3.set_title(f'NNIC - All Events\n(Total: {nnic_total:,} proteins)', 
                  fontsize=14, fontweight='bold')
    
    # Panel 4: Pie chart for NIC + NNIC combined (all events)
    combined_matched = nic_matched + nnic_matched
    combined_nj = nic_nj + nnic_nj
    combined_ir = nic_ir + nnic_ir
    combined_both = nic_both + nnic_both
    combined_total = nic_total + nnic_total
    
    combined_sizes = [combined_matched, combined_nj, combined_ir, combined_both]
    combined_labels = [
        f'Others\n({combined_matched:,})',
        f'Novel Junction\n({combined_nj:,})',
        f'Intron Retention\n({combined_ir:,})',
        f'NJ + IR\n({combined_both:,})'
    ]
    combined_colors = [color_matched_pie, color_nj, color_ir, color_both]
    
    # Filter out zero values
    combined_sizes_filtered = [s for s in combined_sizes if s > 0]
    combined_labels_filtered = [l for l, s in zip(combined_labels, combined_sizes) if s > 0]
    combined_colors_filtered = [c for c, s in zip(combined_colors, combined_sizes) if s > 0]
    
    wedges4, texts4, autotexts4 = ax4.pie(combined_sizes_filtered, labels=combined_labels_filtered, 
                                           colors=combined_colors_filtered, 
                                           autopct='%1.1f%%', startangle=90,
                                           textprops={'fontsize': 14},
                                           pctdistance=0.85)
    
    for autotext in autotexts4:
        autotext.set_fontsize(12)
        autotext.set_fontweight('bold')
        autotext.set_color('white')
    
    for text in texts4:
        text.set_fontsize(14)
        text.set_fontweight('bold')
    
    ax4.set_title(f'NIC + NNIC - All Events\n(Total: {combined_total:,} proteins)', 
                  fontsize=14, fontweight='bold')
    
    # ========================================
    # NEW PANELS: Novel Proteoforms ONLY
    # ========================================
    
    # Panel 5: NIC - Novel Proteoforms Only (3 categories)
    nic_novel_total = nic_nj + nic_ir + nic_both
    
    if nic_novel_total > 0:
        nic_novel_sizes = [nic_nj, nic_ir, nic_both]
        nic_novel_labels = [
            f'Novel Junction\n({nic_nj:,})',
            f'Intron Retention\n({nic_ir:,})',
            f'NJ + IR\n({nic_both:,})'
        ]
        nic_novel_colors = [color_nj_novel, color_ir_novel, color_both_novel]
        
        # Filter out zero values
        nic_novel_sizes_filtered = [s for s in nic_novel_sizes if s > 0]
        nic_novel_labels_filtered = [l for l, s in zip(nic_novel_labels, nic_novel_sizes) if s > 0]
        nic_novel_colors_filtered = [c for c, s in zip(nic_novel_colors, nic_novel_sizes) if s > 0]
        
        wedges5, texts5, autotexts5 = ax5.pie(nic_novel_sizes_filtered, labels=nic_novel_labels_filtered, 
                                               colors=nic_novel_colors_filtered, 
                                               autopct='%1.1f%%', startangle=90,
                                               textprops={'fontsize': 14},
                                               pctdistance=0.85)
        
        for autotext in autotexts5:
            autotext.set_fontsize(12)
            autotext.set_fontweight('bold')
            autotext.set_color('white')
        
        for text in texts5:
            text.set_fontsize(14)
            text.set_fontweight('bold')
        
        ax5.set_title(f'NIC - Novel Proteoforms Only\n(Total: {nic_novel_total:,} proteins)', 
                      fontsize=14, fontweight='bold')
    else:
        ax5.text(0.5, 0.5, 'No Novel\nProteoforms', ha='center', va='center', 
                fontsize=16, fontweight='bold')
        ax5.set_title(f'NIC - Novel Proteoforms Only\n(Total: 0 proteins)', 
                      fontsize=14, fontweight='bold')
    
    # Panel 6: NNIC - Novel Proteoforms Only (3 categories)
    nnic_novel_total = nnic_nj + nnic_ir + nnic_both
    
    if nnic_novel_total > 0:
        nnic_novel_sizes = [nnic_nj, nnic_ir, nnic_both]
        nnic_novel_labels = [
            f'Novel Junction\n({nnic_nj:,})',
            f'Intron Retention\n({nnic_ir:,})',
            f'NJ + IR\n({nnic_both:,})'
        ]
        nnic_novel_colors = [color_nj_novel, color_ir_novel, color_both_novel]
        
        # Filter out zero values
        nnic_novel_sizes_filtered = [s for s in nnic_novel_sizes if s > 0]
        nnic_novel_labels_filtered = [l for l, s in zip(nnic_novel_labels, nnic_novel_sizes) if s > 0]
        nnic_novel_colors_filtered = [c for c, s in zip(nnic_novel_colors, nnic_novel_sizes) if s > 0]
        
        wedges6, texts6, autotexts6 = ax6.pie(nnic_novel_sizes_filtered, labels=nnic_novel_labels_filtered, 
                                               colors=nnic_novel_colors_filtered, 
                                               autopct='%1.1f%%', startangle=90,
                                               textprops={'fontsize': 14},
                                               pctdistance=0.85)
        
        for autotext in autotexts6:
            autotext.set_fontsize(12)
            autotext.set_fontweight('bold')
            autotext.set_color('white')
        
        for text in texts6:
            text.set_fontsize(14)
            text.set_fontweight('bold')
        
        ax6.set_title(f'NNIC - Novel Proteoforms Only\n(Total: {nnic_novel_total:,} proteins)', 
                      fontsize=14, fontweight='bold')
    else:
        ax6.text(0.5, 0.5, 'No Novel\nProteoforms', ha='center', va='center', 
                fontsize=16, fontweight='bold')
        ax6.set_title(f'NNIC - Novel Proteoforms Only\n(Total: 0 proteins)', 
                      fontsize=14, fontweight='bold')
    
    # Panel 7: NIC + NNIC Combined - Novel Proteoforms Only (3 categories)
    combined_novel_total = combined_nj + combined_ir + combined_both
    
    if combined_novel_total > 0:
        combined_novel_sizes = [combined_nj, combined_ir, combined_both]
        combined_novel_labels = [
            f'Novel Junction\n({combined_nj:,})',
            f'Intron Retention\n({combined_ir:,})',
            f'NJ + IR\n({combined_both:,})'
        ]
        combined_novel_colors = [color_nj_novel, color_ir_novel, color_both_novel]
        
        # Filter out zero values
        combined_novel_sizes_filtered = [s for s in combined_novel_sizes if s > 0]
        combined_novel_labels_filtered = [l for l, s in zip(combined_novel_labels, combined_novel_sizes) if s > 0]
        combined_novel_colors_filtered = [c for c, s in zip(combined_novel_colors, combined_novel_sizes) if s > 0]
        
        wedges7, texts7, autotexts7 = ax7.pie(combined_novel_sizes_filtered, labels=combined_novel_labels_filtered, 
                                               colors=combined_novel_colors_filtered, 
                                               autopct='%1.1f%%', startangle=90,
                                               textprops={'fontsize': 14},
                                               pctdistance=0.85)
        
        for autotext in autotexts7:
            autotext.set_fontsize(12)
            autotext.set_fontweight('bold')
            autotext.set_color('white')
        
        for text in texts7:
            text.set_fontsize(14)
            text.set_fontweight('bold')
        
        ax7.set_title(f'NIC + NNIC - Novel Proteoforms Only\n(Total: {combined_novel_total:,} proteins)', 
                      fontsize=14, fontweight='bold')
    else:
        ax7.text(0.5, 0.5, 'No Novel\nProteoforms', ha='center', va='center', 
                fontsize=16, fontweight='bold')
        ax7.set_title(f'NIC + NNIC - Novel Proteoforms Only\n(Total: 0 proteins)', 
                      fontsize=14, fontweight='bold')
    
    # Save in three formats: PNG, SVG, and PDF
    png_file = f'{output_prefix}.png'
    svg_file = f'{output_prefix}.svg'
    pdf_file = f'{output_prefix}.pdf'
    plt.savefig(png_file, dpi=300, bbox_inches='tight')
    plt.savefig(svg_file, format='svg', bbox_inches='tight')
    plt.savefig(pdf_file, format='pdf', bbox_inches='tight')
    print(f"\nVisualization saved to:")
    print(f"  - {png_file}")
    print(f"  - {svg_file}")
    print(f"  - {pdf_file}")
    plt.close()
    
    # Print summary
    print("\n=== Summary by Category (Deduplicated Proteins) ===")
    for cat in categories:
        counts = category_counts[cat]
        total = counts['total']
        matched = counts['matched']
        novel_total = counts['novel_junction'] + counts['intron_retention'] + counts['both']
        print(f"{cat:4s}: {total:6,} proteins | "
              f"Others: {matched:5,} ({100*matched/total if total > 0 else 0:.1f}%) | "
              f"Novel Proteoforms: {novel_total:5,} ({100*novel_total/total if total > 0 else 0:.1f}%)")
        print(f"       Details - NJ: {counts['novel_junction']:5,} | "
              f"IR: {counts['intron_retention']:5,} | "
              f"Both: {counts['both']:5,}")
    
    print(f"\n=== NIC + NNIC Combined Summary ===")
    print(f"Total: {combined_total:,} proteins")
    print(f"  - Others:           {combined_matched:,} ({100*combined_matched/combined_total:.1f}%)")
    print(f"  - Novel Junction:   {combined_nj:,} ({100*combined_nj/combined_total:.1f}%)")
    print(f"  - Intron Retention: {combined_ir:,} ({100*combined_ir/combined_total:.1f}%)")
    print(f"  - NJ + IR:          {combined_both:,} ({100*combined_both/combined_total:.1f}%)")
    
    print(f"\n=== Novel Proteoforms ONLY Summary ===")
    print(f"NIC Novel Proteoforms: {nic_novel_total:,}")
    print(f"  - Novel Junction:   {nic_nj:,} ({100*nic_nj/nic_novel_total if nic_novel_total > 0 else 0:.1f}%)")
    print(f"  - Intron Retention: {nic_ir:,} ({100*nic_ir/nic_novel_total if nic_novel_total > 0 else 0:.1f}%)")
    print(f"  - NJ + IR:          {nic_both:,} ({100*nic_both/nic_novel_total if nic_novel_total > 0 else 0:.1f}%)")
    print(f"\nNNIC Novel Proteoforms: {nnic_novel_total:,}")
    print(f"  - Novel Junction:   {nnic_nj:,} ({100*nnic_nj/nnic_novel_total if nnic_novel_total > 0 else 0:.1f}%)")
    print(f"  - Intron Retention: {nnic_ir:,} ({100*nnic_ir/nnic_novel_total if nnic_novel_total > 0 else 0:.1f}%)")
    print(f"  - NJ + IR:          {nnic_both:,} ({100*nnic_both/nnic_novel_total if nnic_novel_total > 0 else 0:.1f}%)")
    print(f"\nCombined (NIC + NNIC) Novel Proteoforms: {combined_novel_total:,}")
    print(f"  - Novel Junction:   {combined_nj:,} ({100*combined_nj/combined_novel_total if combined_novel_total > 0 else 0:.1f}%)")
    print(f"  - Intron Retention: {combined_ir:,} ({100*combined_ir/combined_novel_total if combined_novel_total > 0 else 0:.1f}%)")
    print(f"  - NJ + IR:          {combined_both:,} ({100*combined_both/combined_novel_total if combined_novel_total > 0 else 0:.1f}%)")

def print_gene_specific_details(results_df, isoform_gtf_dict, protein_data, 
                                ref_internal_exons, ref_donors, ref_acceptors,
                                gene_names=['AIF1', 'PVRIG']):
    """Print detailed information for specific genes of interest with exon-level debugging."""
    print("\n" + "=" * 80)
    print(f"DETAILED ANALYSIS FOR GENES: {', '.join(gene_names)}")
    print("=" * 80)
    
    for gene_name in gene_names:
        print(f"\n{'=' * 80}")
        print(f"Gene: {gene_name}")
        print(f"{'=' * 80}")
        
        # Get all isoforms/proteins for this gene
        gene_data = results_df[results_df['gene_name'] == gene_name].copy()
        
        if len(gene_data) == 0:
            print(f"  ⚠️  No data found for gene {gene_name}")
            continue
        
        # Get representative proteins only
        rep_proteins = gene_data[gene_data['is_representative'] == True].copy()
        
        print(f"\nTotal isoforms: {len(gene_data)}")
        print(f"Unique proteins (deduplicated): {len(rep_proteins)}")
        
        # SQANTI category breakdown
        print(f"\n--- SQANTI Category Distribution ---")
        for cat in ['FSM', 'ISM', 'NIC', 'NNIC']:
            cat_count = len(gene_data[gene_data['sqanti_cat'] == cat])
            if cat_count > 0:
                print(f"  {cat:4s}: {cat_count:3,} isoforms")
        
        # Novel event classification (for representative proteins)
        if len(rep_proteins) > 0:
            print(f"\n--- Novel Event Classification (Deduplicated Proteins) ---")
            
            matched = rep_proteins[(~rep_proteins['has_novel_junction_in_cds']) & 
                                  (~rep_proteins['has_intron_retention_in_cds'])]
            novel_junc = rep_proteins[(rep_proteins['has_novel_junction_in_cds']) & 
                                     (~rep_proteins['has_intron_retention_in_cds'])]
            intron_ret = rep_proteins[(~rep_proteins['has_novel_junction_in_cds']) & 
                                     (rep_proteins['has_intron_retention_in_cds'])]
            both = rep_proteins[(rep_proteins['has_novel_junction_in_cds']) & 
                               (rep_proteins['has_intron_retention_in_cds'])]
            
            print(f"  Matched (Others):                {len(matched):3,} proteins")
            print(f"  Novel Junction only:             {len(novel_junc):3,} proteins")
            print(f"  Intron Retention only:           {len(intron_ret):3,} proteins")
            print(f"  Both (NJ + IR):                  {len(both):3,} proteins")
            
            # Protein length statistics
            print(f"\n--- Protein AA Length Statistics ---")
            if rep_proteins['protein_aa_length'].notna().sum() > 0:
                print(f"  Min:    {rep_proteins['protein_aa_length'].min():.0f} AA")
                print(f"  Max:    {rep_proteins['protein_aa_length'].max():.0f} AA")
                print(f"  Mean:   {rep_proteins['protein_aa_length'].mean():.1f} AA")
                print(f"  Median: {rep_proteins['protein_aa_length'].median():.1f} AA")
            
            # Detailed protein list
            print(f"\n--- Representative Protein Details ---")
            for idx, row in rep_proteins.iterrows():
                event_type = "Others"
                if row['has_novel_junction_in_cds'] and row['has_intron_retention_in_cds']:
                    event_type = "NJ+IR"
                elif row['has_novel_junction_in_cds']:
                    event_type = "Novel Junction"
                elif row['has_intron_retention_in_cds']:
                    event_type = "Intron Retention"
                
                print(f"\n  Protein: {row['protein_id']}")
                print(f"    Isoform:        {row['isoform_id']}")
                print(f"    SQANTI Cat:     {row['sqanti_cat']}")
                print(f"    Event Type:     {event_type}")
                print(f"    AA Length:      {row['protein_aa_length']:.0f} AA" if pd.notna(row['protein_aa_length']) else "    AA Length:      N/A")
                print(f"    Chromosome:     {row['chrom']} ({row['strand']})")
                print(f"    Exons:          {row['num_exons']}")
                print(f"    Novel Junctions: {row['num_novel_junctions']}")
                print(f"    Ref Junctions:   {row['num_ref_junctions']}")
        
        # EXON-LEVEL DEBUGGING
        print(f"\n{'=' * 80}")
        print(f"EXON-LEVEL INTRON RETENTION DEBUGGING")
        print(f"{'=' * 80}")
        
        for idx, row in gene_data.iterrows():
            isoform_id = row['isoform_id']
            protein_id = row['protein_id']
            
            print(f"\n{'─' * 80}")
            print(f"Isoform: {isoform_id}")
            print(f"SQANTI: {row['sqanti_cat']} | Exons: {row['num_exons']} | "
                  f"Novel Junctions: {row['num_novel_junctions']}")
            
            if isoform_id not in isoform_gtf_dict:
                print("  ⚠️  Not found in GTF")
                continue
            
            gtf_data = isoform_gtf_dict[isoform_id]
            exons = gtf_data['exons']
            chrom = gtf_data['chrom']
            strand = gtf_data['strand']
            
            print(f"\nExon coordinates (0-based):")
            for i, (start, end) in enumerate(exons):
                exon_type = "FIRST" if i == 0 else ("LAST" if i == len(exons)-1 else "INTERNAL")
                print(f"  Exon {i+1}: {start:,}-{end:,} ({end-start} bp) [{exon_type}]")
            
            # Check if protein has CDS data
            if pd.isna(protein_id):
                print("\n  ⚠️  No complete ORF protein")
                continue
            
            if protein_id not in protein_data:
                print(f"\n  ⚠️  Protein not in protein_data")
                continue
            
            prot_info = protein_data[protein_id]
            cds_regions = prot_info['cds_regions']
            
            print(f"\nCDS regions (0-based, INCLUDING stop codon):")
            for i, (cds_start, cds_end) in enumerate(cds_regions):
                if i == len(cds_regions) - 1:
                    print(f"  CDS {i+1}: {cds_start:,}-{cds_end:,} ({cds_end-cds_start} bp) [last, includes stop codon]")
                else:
                    print(f"  CDS {i+1}: {cds_start:,}-{cds_end:,} ({cds_end-cds_start} bp)")
            
            cds_start_overall = cds_regions[0][0]
            cds_end_overall = cds_regions[-1][1]  # INCLUDE stop codon
            print(f"\nOverall CDS: {cds_start_overall:,} to {cds_end_overall:,} (includes stop codon)")
            print(f"Buffer zone: {cds_start_overall + NT_BUFFER:,} to {cds_end_overall - NT_BUFFER:,} (±{NT_BUFFER}bp)")
            
            # Get gene-specific reference sets
            ref_key = (gene_name, chrom, strand)
            gene_ref_internal_exons = ref_internal_exons.get(ref_key, set())
            gene_ref_donors = ref_donors.get(ref_key, set())
            gene_ref_acceptors = ref_acceptors.get(ref_key, set())
            
            # Check internal exons for intron retention
            if len(exons) < 3:
                print(f"\n  ⚠️  Only {len(exons)} exons - no internal exons possible")
                continue
            
            print(f"\n=== Checking INTERNAL exons for intron retention (gene: {gene_name}) ===")
            for i in range(1, len(exons) - 1):
                exon_start, exon_end = exons[i]
                
                print(f"\nInternal Exon {i+1}: {exon_start:,}-{exon_end:,}")
                
                # Check if in gene-specific reference
                in_reference = (exon_start, exon_end) in gene_ref_internal_exons
                print(f"  In reference internal exons ({gene_name}): {in_reference}")
                
                if in_reference:
                    print(f"  ✓ Known internal exon for {gene_name} - NOT intron retention")
                    continue
                
                # Check gene-specific splice sites
                donor_known = exon_end in gene_ref_donors
                acceptor_known = exon_start in gene_ref_acceptors
                
                print(f"  Donor site ({exon_end:,}):     Known={donor_known} (in {gene_name})")
                print(f"  Acceptor site ({exon_start:,}): Known={acceptor_known} (in {gene_name})")
                
                if not (donor_known and acceptor_known):
                    print(f"  ✗ Novel splice sites - NOT intron retention")
                    continue
                
                print(f"  ✓ Both splice sites are KNOWN in {gene_name}")
                
                # Check if endpoints in CDS
                print(f"\n  Checking if endpoints are in CDS with buffer:")
                
                for endpoint_name, endpoint in [("Acceptor/Start", exon_start), ("Donor/End", exon_end)]:
                    print(f"\n    {endpoint_name} ({endpoint:,}):")
                    
                    # Check if in any CDS region (NO stop codon adjustment)
                    in_cds = False
                    for cds_start, cds_end in cds_regions:
                        if cds_start <= endpoint <= cds_end:
                            in_cds = True
                            print(f"      In CDS region {cds_start:,}-{cds_end:,}: YES")
                            break
                    
                    if not in_cds:
                        print(f"      In CDS: NO")
                        continue
                    
                    # Check buffer
                    in_buffer = (endpoint >= cds_start_overall + NT_BUFFER and 
                                endpoint <= cds_end_overall - NT_BUFFER)
                    
                    dist_from_start = endpoint - cds_start_overall
                    dist_from_end = cds_end_overall - endpoint
                    
                    print(f"      Distance from CDS start: {dist_from_start} bp")
                    print(f"      Distance from CDS end:   {dist_from_end} bp")
                    print(f"      In buffer (≥{NT_BUFFER}bp from both ends): {in_buffer}")
                    
                    if in_buffer:
                        print(f"\n  ✓✓✓ INTRON RETENTION DETECTED!")
                        print(f"      Internal exon NOT in reference for {gene_name}")
                        print(f"      Both endpoints use known splice sites in {gene_name}")
                        print(f"      Endpoint {endpoint_name} in CDS with valid buffer")
                        break
                else:
                    print(f"\n  ✗ Endpoints not in CDS buffer zone")
    
    print("\n" + "=" * 80)

def process_dataset(dataset_name, dataset_config, ref_junctions, ref_internal_exons, 
                   ref_donors, ref_acceptors, min_aa_length_thresholds=[50, 100, 200]):
    """Process a single dataset with the given configuration."""
    
    print("\n" + "=" * 80)
    print(f"PROCESSING DATASET: {dataset_name}")
    print("=" * 80)
    
    # Set random seeds for reproducibility
    random.seed(42)
    np.random.seed(42)
    
    # Create output directory
    output_dir = dataset_config['OUTPUT_DIR']
    os.makedirs(output_dir, exist_ok=True)
    print(f"\nOutput directory: {output_dir}/")
    
    print("=" * 80)
    print(f"Novel Junction & Intron Retention Analysis - {dataset_name}")
    print(f"Complete ORFs + CDS ± {NT_BUFFER}bp Buffer")
    print(f"Protein Length Filters: {min_aa_length_thresholds} AA")
    print(f"Deduplication: By amino acid sequence")
    print("=" * 80)
    
    # Sanity check: Verify all files use genomic coordinates
    if not verify_coordinate_systems(dataset_config):
        print(f"\n❌ CRITICAL ERROR: Coordinate system mismatch detected for {dataset_name}!")
        print("❌ Please ensure all files (GTF/GFF3) use genomic coordinates.")
        return
    
    # Create transcript-level plot (only once, independent of protein filtering)
    plot_transcript_level_counts(dataset_config['SQANTI_FILE'], 
                                 os.path.join(output_dir, 'transcript_level_counts'))
    
    # Parse isoform GTF
    isoform_gtf_dict = parse_isoform_gtf(dataset_config['ISOFORM_GTF'])
    
    # Loop through each protein length threshold
    for min_aa_length in min_aa_length_thresholds:
        print("\n" + "=" * 80)
        print(f"ANALYSIS WITH MINIMUM PROTEIN LENGTH: ≥{min_aa_length} AA")
        print("=" * 80)
        
        # Parse representative proteins (COMPLETE ORFs ONLY) with length filter
        complete_orf_proteins, protein_aa_lengths, protein_sequences = parse_protein_sequences(
            dataset_config['PROTEIN_PEP'], min_aa_length=min_aa_length
        )
        
        if len(complete_orf_proteins) == 0:
            print(f"\n⚠️  WARNING: No complete ORF proteins ≥{min_aa_length} AA found! Skipping this threshold.")
            continue
        
        # Load protein data
        protein_data = parse_gff3_protein_file(dataset_config['PROTEIN_GFF3'], complete_orf_proteins)
        
        if len(protein_data) == 0:
            print(f"\n⚠️  WARNING: No protein CDS data loaded for ≥{min_aa_length} AA! Skipping this threshold.")
            continue
        
        # Build fast lookup map
        isoform_to_protein = build_protein_to_isoform_map(protein_data)
        
        # Analyze novel junctions and intron retention
        results_df = identify_novel_junctions(
            dataset_config['SQANTI_FILE'], protein_data, isoform_gtf_dict, isoform_to_protein,
            ref_junctions, ref_internal_exons, ref_donors, ref_acceptors, 
            protein_aa_lengths, protein_sequences
        )
        
        # Create output subdirectory for this threshold
        threshold_suffix = f"_minAA{min_aa_length}"
        
        # Save results (protein_sequence column will be included)
        output_tsv = os.path.join(output_dir, f'novel_junction_results_protein_level{threshold_suffix}.tsv')
        results_df.to_csv(output_tsv, sep='\t', index=False)
        print(f"\nResults saved to {output_tsv}")
        
        # Create visualization with threshold suffix
        plot_results(results_df, os.path.join(output_dir, f'novel_junction_analysis{threshold_suffix}'), 
                    min_aa_length=min_aa_length)
        
        # Subsample proteins for manual review
        subsample_df = subsample_proteins_by_gene(results_df, n_per_group=5, min_isoforms=2, max_isoforms=4, 
                                                 output_dir=output_dir, suffix=threshold_suffix)
        
        # Print gene-specific details (only for first threshold to avoid repetition)
        if min_aa_length == min_aa_length_thresholds[0]:
            print_gene_specific_details(
                results_df, isoform_gtf_dict, protein_data,
                ref_internal_exons, ref_donors, ref_acceptors,
                gene_names=['AIF1', 'PVRIG']
            )
    
    print("\n" + "=" * 80)
    print(f"Analysis complete for {dataset_name}!")
    print(f"All results saved in: {output_dir}/")
    print(f"Thresholds analyzed: {min_aa_length_thresholds} AA")
    print("=" * 80)


def main(datasets_to_process=['ALS', 'PBMC'], min_aa_length_thresholds=[50, 100, 200]):
    """
    Main execution function that processes multiple datasets.
    
    Args:
        datasets_to_process: List of dataset names to process (e.g., ['ALS', 'PBMC'])
        min_aa_length_thresholds: List of minimum AA length thresholds
    """
    
    print("\n" + "=" * 80)
    print("MULTI-DATASET PROTEOFORM ANALYSIS PIPELINE")
    print(f"Datasets: {', '.join(datasets_to_process)}")
    print("=" * 80)
    
    # Build reference data ONCE (shared across all datasets)
    print("\n" + "=" * 80)
    print("BUILDING REFERENCE DATA (gene-specific, shared across datasets)")
    print("=" * 80)
    
    # Build reference junction set
    ref_junctions = build_reference_junction_set(REF_GTF)
    
    if len(ref_junctions) == 0:
        print("\n⚠️  ERROR: No reference junctions loaded. Cannot proceed.")
        return
    
    # Build reference internal exon set and splice sites - GENE-SPECIFIC
    ref_internal_exons, ref_donors, ref_acceptors = parse_gencode_gtf(REF_GTF)
    
    # Process each dataset
    for dataset_name in datasets_to_process:
        if dataset_name not in DATASETS:
            print(f"\n⚠️  WARNING: Dataset '{dataset_name}' not found in configuration. Skipping.")
            continue
        
        dataset_config = DATASETS[dataset_name]
        
        try:
            process_dataset(dataset_name, dataset_config, ref_junctions, 
                          ref_internal_exons, ref_donors, ref_acceptors, 
                          min_aa_length_thresholds)
        except Exception as e:
            print(f"\n❌ ERROR processing {dataset_name}: {e}")
            raise
    
    print("\n" + "=" * 80)
    print("ALL DATASETS PROCESSED!")
    print(f"Datasets analyzed: {', '.join(datasets_to_process)}")
    print(f"Thresholds analyzed: {min_aa_length_thresholds} AA")
    print("=" * 80)


if __name__ == '__main__':
    # Process both datasets with default settings
    main(datasets_to_process=['ALS', 'PBMC'], min_aa_length_thresholds=[50, 100, 200])
    
    # Alternative: Process only one dataset
    # main(datasets_to_process=['ALS'], min_aa_length_thresholds=[50, 100, 200])

In [ ]:
# Paper figures: top, unique complete-ORF proteins per SQLIKE category (representatives after
# deduplication by protein sequence), reference-matched vs novel; bottom, the novel proteoforms of
# NIC + NNIC by event type. Same counts as plot_results() panels 1 and 7.
import matplotlib
import matplotlib.patches
import matplotlib.ticker
matplotlib.rcParams['pdf.fonttype'] = 42

THRESHOLDS = [50, 100, 200]
CATS = ['FSM', 'ISM', 'NIC', 'NNIC']
COL_MATCHED, COL_NOVEL = '#95a5a6', '#e74c3c'
COL_NJ, COL_IR, COL_BOTH = '#3498db', '#f39c12', '#8e44ad'   # 'both' not red: red is the bars' novel


def proteoform_counts(dataset):
    rows = []
    for t in THRESHOLDS:
        df = pd.read_csv(os.path.join(DATASETS[dataset]['OUTPUT_DIR'],
                                      f'novel_junction_results_protein_level_minAA{t}.tsv'), sep='\t')
        rep = df[df['is_representative'] == True]
        nj, ir = rep['has_novel_junction_in_cds'], rep['has_intron_retention_in_cds']
        rep = rep.assign(event=np.select([nj & ir, nj, ir], ['both', 'novel_junction', 'intron_retention'],
                                         'matched'))
        for cat in CATS:
            ev = rep.loc[rep['sqanti_cat'] == cat, 'event'].value_counts()
            rows.append({'dataset': dataset, 'min_aa': t, 'sqanti_cat': cat,
                         **{e: int(ev.get(e, 0)) for e in ['matched', 'novel_junction', 'intron_retention', 'both']}})
    out = pd.DataFrame(rows)
    out['novel'] = out[['novel_junction', 'intron_retention', 'both']].sum(axis=1)
    out['total'] = out['matched'] + out['novel']
    return out


def plot_paper_figure(counts, pdf):
    # drawn at its printed size (the manuscript text width), so the font sizes are the printed ones
    fig = plt.figure(figsize=(7, 4.6))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.25, 1], hspace=0.45, wspace=0.35,
                          left=0.09, right=0.99, top=0.95, bottom=0.13)
    for j, t in enumerate(THRESHOLDS):
        c = counts[counts['min_aa'] == t].set_index('sqanti_cat').loc[CATS]
        ax = fig.add_subplot(gs[0, j])
        ax.bar(CATS, c['matched'], 0.7, color=COL_MATCHED, label='Reference Matched Proteoform')
        ax.bar(CATS, c['novel'], 0.7, bottom=c['matched'], color=COL_NOVEL, label='Novel Proteoform')
        ymax = c['total'].max()
        for i, cat in enumerate(CATS):
            for y0, h in ((0, c.loc[cat, 'matched']), (c.loc[cat, 'matched'], c.loc[cat, 'novel'])):
                if h > 0.07 * ymax:     # label only segments tall enough to hold it
                    ax.text(i, y0 + h / 2, f'{h:,}', ha='center', va='center', color='white',
                            fontsize=5, fontweight='bold')
        ax.set_title(f'≥{t} AA', fontsize=8, fontweight='bold')
        ax.tick_params(labelsize=6)
        ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f'{v / 1000:g}k' if v else '0'))
        ax.grid(axis='y', alpha=0.3)
        ax.set_axisbelow(True)
        if j == 0:
            ax.set_ylabel('Number of Unique Proteins', fontsize=7)

        nn = c.loc[['NIC', 'NNIC'], ['novel_junction', 'intron_retention', 'both']].sum()
        sizes = [nn['novel_junction'], nn['intron_retention'], nn['both']]
        ax = fig.add_subplot(gs[1, j])
        _, _, autotexts = ax.pie(sizes, colors=[COL_NJ, COL_IR, COL_BOTH], startangle=90, counterclock=False,
                                 autopct=lambda v: f'{v:.1f}%' if v >= 3 else '', pctdistance=0.68,
                                 textprops={'fontsize': 6, 'fontweight': 'bold', 'color': 'white'},
                                 wedgeprops={'linewidth': 0.5, 'edgecolor': 'white'})
        ax.set_title(f'Novel proteoforms: {sum(sizes):,}', fontsize=7, fontweight='bold', pad=2)
        ax.text(0.5, -0.08, f'NJ {sizes[0]:,} · IR {sizes[1]:,} · both {sizes[2]:,} ({100 * sizes[2] / sum(sizes):.1f}%)',
                transform=ax.transAxes, ha='center', va='top', fontsize=5.5)
    bar_handles = [matplotlib.patches.Patch(color=COL_MATCHED, label='Reference Matched Proteoform'),
                   matplotlib.patches.Patch(color=COL_NOVEL, label='Novel Proteoform')]
    pie_handles = [matplotlib.patches.Patch(color=COL_NJ, label='Novel Junction (NJ)'),
                   matplotlib.patches.Patch(color=COL_IR, label='Intron Retention (IR)'),
                   matplotlib.patches.Patch(color=COL_BOTH, label='Novel Junction + Intron Retention (both)')]
    # one legend per row: the bars' legend under the bars, the pies' (novel proteoforms by event type) under the pies
    fig.legend(handles=bar_handles, loc='center', bbox_to_anchor=(0.54, 0.485), ncol=2,
               fontsize=5.5, frameon=False, handlelength=1, columnspacing=1)
    fig.legend(handles=pie_handles, loc='lower center', bbox_to_anchor=(0.54, 0.0), ncol=3,
               fontsize=5.5, frameon=False, handlelength=1, columnspacing=1, title='Novel proteoforms by event type',
               title_fontsize=5.5)
    fig.savefig(pdf)
    plt.close(fig)
    print('wrote', pdf)


all_counts = []
for ds, fig_name in (('PBMC', 'SuppFig_proteoforms_PBMC.pdf'), ('ALS', 'SuppFig_proteoforms_FTD.pdf')):
    counts = proteoform_counts(ds)
    plot_paper_figure(counts, fig_name)
    all_counts.append(counts)
all_counts = pd.concat(all_counts)
all_counts.to_csv('proteoform_counts.tsv', sep='\t', index=False)

# The numbers the text quotes (results: novel share of all unique proteins; NIC + NNIC composition)
summary = all_counts.groupby(['dataset', 'min_aa'])[['total', 'novel', 'novel_junction',
                                                      'intron_retention', 'both']].sum()
summary['pct_novel'] = (100 * summary['novel'] / summary['total']).round(1)
for e in ['novel_junction', 'intron_retention', 'both']:
    summary[f'pct_{e}'] = (100 * summary[e] / summary['novel']).round(1)
summary
